In [4]:
import common
from db import get_engine
engine = get_engine()

Microsoft SQL Server 2022 (RTM-CU27-GDR) (KB5126934) - 16.0.
Logged in as: datacube_analytics
     name
0  bronze
1    gold
2  silver


* Xam cedvellerin yuklenmesi

In [6]:
airports_raw=common.pd.read_csv("/workspaces/Flight-delay-analysis-E2E/data/Flight/airports.csv")
airlines_raw=common.pd.read_csv("/workspaces/Flight-delay-analysis-E2E/data/Flight/airlines.csv")
flights_raw=common.pd.read_parquet('/workspaces/Flight-delay-analysis-E2E/data/Flight/flights.parquet')
weather_raw=common.pd.read_parquet('/workspaces/Flight-delay-analysis-E2E/data/api/weather_2015_all.parquet')

: 

In [4]:
print("--- FLIGHTS ---")
print(flights_raw.info())
print(flights_raw.head(2))

print("\n--- AIRPORTS ---")
print(airports_raw.info())
print(airports_raw.head(2))

print("\n--- AIRLINES ---")
print(airlines_raw.info())
print(airlines_raw.head(2))

print("\n--- WEATHER ---")
print(weather_raw.info())
print(weather_raw.head(2))

--- FLIGHTS ---
<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 31 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int16  
 1   MONTH                int8   
 2   DAY                  int8   
 3   DAY_OF_WEEK          int8   
 4   AIRLINE              string 
 5   FLIGHT_NUMBER        int32  
 6   TAIL_NUMBER          string 
 7   ORIGIN_AIRPORT       string 
 8   DESTINATION_AIRPORT  string 
 9   SCHEDULED_DEPARTURE  int16  
 10  DEPARTURE_TIME       float32
 11  DEPARTURE_DELAY      float32
 12  TAXI_OUT             float32
 13  WHEELS_OFF           float32
 14  SCHEDULED_TIME       float32
 15  ELAPSED_TIME         float32
 16  AIR_TIME             float32
 17  DISTANCE             int16  
 18  WHEELS_ON            float32
 19  TAXI_IN              float32
 20  SCHEDULED_ARRIVAL    int16  
 21  ARRIVAL_TIME         float32
 22  ARRIVAL_DELAY        float32
 23  DIVERTED             int8  

* 5 reqemli aeroport kodlarinin 3 herfli koda cevrilmesi

In [3]:


# 1. Rəsmi BTS (DOT Airport ID -> IATA) xəritəsini oxuyuruq
bts_mapping_url = "https://gist.githubusercontent.com/patil-charudatta/e5cb878df0288898114f2e54eb8ff9c9/raw/dot_airport_id_mapping.csv"
df_bts = common.pd.read_csv(bts_mapping_url, dtype=str)

# 2. Xəritələmə lüğətini yaradırıq (Code -> IATA)
dot_to_iata = dict(zip(df_bts['Code'].str.strip(), df_bts['IATA'].str.strip()))

# 3. flights_raw cədvəlindəki sütunları təmizləyib birbaşa əvəzləyirik
flights_raw['ORIGIN_AIRPORT'] = flights_raw['ORIGIN_AIRPORT'].astype(str).str.strip().replace(dot_to_iata)
flights_raw['DESTINATION_AIRPORT'] = flights_raw['DESTINATION_AIRPORT'].astype(str).str.strip().replace(dot_to_iata)

# 4. Yoxlama: Hələ də çevrilməmiş rəqəmli kod qaldımı?
remaining_origin = flights_raw[flights_raw['ORIGIN_AIRPORT'].str.isnumeric()]['ORIGIN_AIRPORT'].unique()
remaining_dest = flights_raw[flights_raw['DESTINATION_AIRPORT'].str.isnumeric()]['DESTINATION_AIRPORT'].unique()

print(f"ORIGIN üzrə qalan rəqəmli kod sayı: {len(remaining_origin)}")
print(f"DESTINATION üzrə qalan rəqəmli kod sayı: {len(remaining_dest)}")

NameError: name 'common' is not defined